# Notebook 2: Observabilidad con Langfuse

## Objetivos de aprendizaje
- Entender por qué la **observabilidad** es imprescindible para LLMs en producción (los logs no son suficientes)
- Conocer el **modelo de datos** de Langfuse: **Trace → Observation** (Span, Generation, Event)
- Configurar Langfuse Cloud y verificar la conexión
- Instrumentar el agente TechShop con el decorador **`@observe`** (Langfuse SDK v4)
- Enriquecer trazas con **`propagate_attributes`** (user_id, session_id, metadata)
- Entender las **sesiones** como agrupación de trazas de una conversación
- Generar trazas, analizarlas en el dashboard **y por API**
- Conocer las **4 categorías de métricas LLMOps**: operacionales, coste, calidad, uso

## ¿Por qué este notebook?
En el Notebook 1 construimos un agente que "parece que funciona". Pero no podemos responder preguntas críticas: ¿usó las herramientas o inventó la respuesta? ¿cuántos tokens consumió? ¿cuánto costó? ¿qué herramienta devolvió datos vacíos? **La observabilidad transforma la incertidumbre en datos accionables.**

> **Referencia principal:** [Langfuse Documentation](https://langfuse.com/docs) · [Langfuse Python SDK v4](https://langfuse.com/docs/observability/sdk/instrumentation)

---

## Parte 0: Teoría — ¿Por qué los logs no son suficientes?

### El problema fundamental

En software clásico, un bug produce un **stacktrace explícito**:
```bash
ERROR: NullPointerException at UserService.java:42
→ Sabes exactamente qué falló, dónde y por qué.
→ Lo ves en el log. Lo arreglas.
```

Con LLMs, los fallos son **semánticos** — y completamente invisibles en los logs:
```bash
# LOG tradicional (lo que tienes sin observabilidad):
2025-01-15 10:23:45 INFO  Agent received query: "¿Tenéis portátiles para diseño gráfico?"
2025-01-15 10:23:47 INFO  Agent response generated (350 tokens)
→ Todo parece correcto en el log.
→ Pero la respuesta dice "Te recomiendo el MacBook Pro M3..."
→ TechShop NO vende MacBook. El agente lo INVENTÓ.
→ El log no tiene ningún indicio de que algo fue mal.
```

**¿Qué información falta en el log?**
- ¿El agente llamó a `search_catalog`? ¿O inventó la respuesta sin consultar el catálogo?
- ¿Qué devolvió la herramienta? ¿Resultados reales o vacíos?
- ¿Cuántas veces llamó al LLM? ¿Cuántos tokens consumió?
- ¿Cuánto costó esta respuesta en dinero real?

> **Los logs te dicen QUE pasó. La observabilidad te dice POR QUÉ pasó.** Para un agente LLM, la diferencia es crítica: necesitas ver el flujo completo de razonamiento del modelo, no solo el input y el output.

### Observabilidad = poder responder preguntas sobre tu agente en producción

| Categoría | Preguntas | Sin observabilidad | Con observabilidad |
|-----------|-----------|--------------------|--------------------|
| **Funcional** | ¿Qué herramientas llamó? ¿En qué orden? | "No sé" | Traza con cada span |
| **Calidad** | ¿Inventó información? ¿Usó datos reales? | "Parece que sí" | Veo qué devolvió la herramienta vs. qué dijo el agente |
| **Coste** | ¿Cuántos tokens consumió? ¿Cuánto costó? | "No sé, pero la factura de AWS subió" | $0.003 por request, 600 tokens promedio |
| **Rendimiento** | ¿Cuánto tarda? ¿Dónde está el cuello de botella? | "A veces va lento" | LLM call #2 tarda 800ms (el 70% del tiempo total) |
| **Uso** | ¿Qué preguntan más los usuarios? | "Ni idea" | 40% catálogo, 30% FAQs, 20% fuera de alcance |
| **Seguridad** | ¿Alguien intentó hacer prompt injection? | "Espero que no" | 3 intentos detectados esta semana |

> No es logging mejorado. Es una capa de **inteligencia operacional** que te permite entender, depurar y mejorar tu agente de forma continua.

### El modelo de datos: Trace → Observation (Span, Generation, Event)

Langfuse organiza la información en un **árbol jerárquico**, construido sobre OpenTelemetry:

```mermaid
graph TD
    T["TRACE<br/>(una request completa del usuario)"]
    T --> S1["SPAN: input_guardrail<br/>duration: 5ms, result: safe"]
    T --> S2["SPAN: agent_processing"]
    T --> S3["SPAN: output_guardrail<br/>duration: 3ms, result: valid"]

    S2 --> G1["GENERATION: LLM call #1<br/>model: claude-sonnet<br/>tokens_in: 150, tokens_out: 30<br/>cost: $0.0008"]
    S2 --> S2a["SPAN: tool.search_catalog<br/>input: 'portátiles' → 2 productos"]
    S2 --> G2["GENERATION: LLM call #2<br/>model: claude-sonnet<br/>tokens_in: 400, tokens_out: 200<br/>cost: $0.0023"]
```

**Diferencias clave entre tipos de observation:**

| Concepto | Qué representa | Tiene duración | Tiene tokens/coste | Ejemplo |
|----------|---------------|:-:|:-:|---------|
| **Trace** | Request completa del usuario | Sí (inicio a fin) | Sí (suma de todo) | "¿Qué portátiles tenéis?" |
| **Span** | Operación lógica dentro del trace | Sí | No | Llamada a `search_catalog` |
| **Generation** | Llamada concreta al modelo LLM | Sí | Sí | Claude genera respuesta |
| **Event** | Punto puntual (marca temporal) | No | No | "Guardrail triggered" |

Las **observations** (span, generation, event) se anidan dentro de un trace formando un árbol. Cada observation puede tener observations hijas, lo que permite representar flujos complejos.

**Regla fundamental:** La primera función `@observe` en la cadena de llamadas crea un **trace**. Todas las funciones `@observe` llamadas dentro crean **spans** hijos de ese trace. Esto sucede automáticamente sin configuración adicional.

> **Referencia:** [Langfuse — Data Model](https://langfuse.com/docs/observability/data-model) · [Langfuse — Observation Types](https://langfuse.com/docs/observability/features/observation-types)

### OpenTelemetry: la base del SDK v4

Langfuse v4 está construido **sobre OpenTelemetry (OTEL)** de forma nativa. Esto tiene implicaciones prácticas importantes:

| Ventaja OTEL | Impacto práctico |
|--------------|----------------|
| **Contexto automático** | Las observaciones anidadas forman el árbol automáticamente sin configuración extra |
| **Propagación de atributos** | `user_id`, `session_id`, `metadata` se propagan a TODAS las observaciones hijas |
| **Interoperabilidad** | Strands Agents emite spans OTEL que Langfuse captura automáticamente |
| **Estándar abierto** | Puedes enviar las trazas simultáneamente a Langfuse y a otras herramientas (Datadog, Jaeger) |

> **Importante:** Cuando llamas a `get_client()`, Langfuse registra un proveedor OTEL global. Strands Agents emite spans OTEL nativamente — por eso sus tool calls y LLM calls aparecen en Langfuse **sin código adicional**.

> **Referencia:** [OpenTelemetry Semantic Conventions for GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/) · [Langfuse — OTEL Integration](https://langfuse.com/integrations/native/opentelemetry)

### ¿Qué es Langfuse?

[**Langfuse**](https://langfuse.com) es una plataforma **open-source** de observabilidad y gestión para aplicaciones LLM. Ofrece:

| Capacidad | Descripción | Notebook del curso |
|-----------|------------|-------------------|
| **Tracing** | Captura el flujo completo de cada request (Trace → Spans → Generations) | NB 2 (este) |
| **Sessions** | Agrupa trazas de una misma conversación (`session_id`) | NB 2 (este) |
| **Users** | Identifica quién está usando el agente (`user_id`) | NB 2 (este) |
| **Scores** | Evalúa calidad de respuestas (manual, heurístico, LLM-as-judge) | NB 4 |
| **Prompt Management** | Versionado de prompts con labels y rollback | NB 3 |
| **Dashboards** | Métricas agregadas (latencia, coste, calidad) | NB 2 (este) |

**¿Por qué Langfuse y no otra herramienta?**
- **Open source** — puedes self-hostear con Docker o usar el SaaS (Cloud)
- **Plan gratuito** suficiente para aprendizaje (50K observaciones/mes)
- **SDK Python nativo** con decorador `@observe` de una línea
- **Multi-provider** — funciona con Bedrock, OpenAI, Anthropic, etc.
- **Integración OTEL** — recibe trazas OpenTelemetry estándar de Strands Agents
- **Una sola plataforma** para tracing + prompts + evaluación


> En este curso usamos Langfuse porque cubre tracing + prompts + evals en una sola herramienta gratuita. En un entorno empresarial, la elección dependerá del stack existente.

> **Referencia:** [Langfuse — Why Langfuse](https://langfuse.com/docs) · [Langfuse — Self-hosting](https://langfuse.com/docs/deployment/self-host)

## Tutorial: Configurar Langfuse Cloud (5 minutos)

### Paso 1: Crear cuenta
1. Ve a **[cloud.langfuse.com](https://cloud.langfuse.com)**
2. Click en **Sign Up** (puedes usar GitHub, Google o email)
3. El plan **Hobby** es gratuito: 50K observaciones/mes, retención 30 días

### Paso 2: Crear proyecto
1. Una vez dentro, click en **New Project**
2. Nombre: `techshop-llmops`
3. Click **Create**

### Paso 3: Obtener API Keys
1. Ve a **Settings** en el menú lateral
2. Click en **API Keys**
3. Click en **Create API Key**
4. Copia los 3 valores:
   - `Public Key` -> `LANGFUSE_PUBLIC_KEY`
   - `Secret Key` -> `LANGFUSE_SECRET_KEY`
   - `Host` -> `https://cloud.langfuse.com`

### Paso 4: Actualizar tu .env
```
LANGFUSE_PUBLIC_KEY=pk-lf-xxxxxxxx
LANGFUSE_SECRET_KEY=sk-lf-xxxxxxxx
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

> **Importante:** La variable de host es `LANGFUSE_BASE_URL` (no `LANGFUSE_HOST`). Nunca compartas tus Secret Keys. El archivo `.env` ya está en `.gitignore`.

---

## Parte 1: Setup y conexión a Langfuse

In [1]:
import strands
import langfuse
import boto3

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv

# Carga variables de entorno desde .env (busca subiendo desde el directorio actual)
load_dotenv(find_dotenv(usecwd=True))

# Verificar AWS
assert os.getenv("AWS_ACCESS_KEY_ID"), "Falta AWS_ACCESS_KEY_ID"

# Verificar Langfuse (variable de host: LANGFUSE_BASE_URL, no LANGFUSE_HOST)
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "Falta LANGFUSE_PUBLIC_KEY - sigue el tutorial arriba"
assert os.getenv("LANGFUSE_SECRET_KEY"), "Falta LANGFUSE_SECRET_KEY"
assert os.getenv("LANGFUSE_BASE_URL"), "Falta LANGFUSE_BASE_URL (ej: https://cloud.langfuse.com)"

print("[OK] Variables de entorno cargadas")

[OK] Variables de entorno cargadas


### 1.2 Conectar con Langfuse

El SDK de Langfuse se inicializa automáticamente leyendo las variables de entorno. Internamente, crea un **cliente HTTP** que envía las trazas al servidor de Langfuse de forma **asíncrona** (no bloquea tu código).

> **Referencia:** [Langfuse Python SDK — Initialization](https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client)

In [3]:
from langfuse import get_client


# get_client() retorna el cliente Langfuse singleton.
# Lee automáticamente LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY y LANGFUSE_BASE_URL.
langfuse_client = get_client()


# Verificar conexión
try:
    if langfuse_client.auth_check():
        base_url = os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
        print(f"[OK] Conectado a Langfuse: {base_url}")
    else:
        print("[ERROR] auth_check() devolvió False. Verifica las credenciales.")
except Exception as e:
    print(f"[ERROR] Error de conexión: {e}")
    print("   Verifica LANGFUSE_PUBLIC_KEY y LANGFUSE_SECRET_KEY en tu .env")

[OK] Conectado a Langfuse: https://cloud.langfuse.com


**¿Qué hace `get_client()` internamente?**

1. Lee `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` y `LANGFUSE_BASE_URL` del entorno
2. Crea un **singleton** global — siempre devuelve el mismo cliente en toda la aplicación
3. Registra un **proveedor OpenTelemetry** — las librerías con soporte OTEL nativo (como Strands Agents) envían sus spans automáticamente
4. Crea un **buffer en memoria** donde se acumulan las observaciones
5. Envía las trazas al servidor en **batches** asíncronos (no bloquea tu código)
6. `auth_check()` verifica las credenciales contra el servidor de Langfuse

> **Importante en notebooks:** En scripts y servidores, Langfuse envía trazas automáticamente. En notebooks, llamamos `lf_client.flush()` explícitamente para forzar el envío antes de ir al dashboard.

---

## Parte 2: Tu primera traza con `@observe`

### El decorador `@observe` — Tu herramienta principal

El decorador [`@observe`](https://langfuse.com/docs/observability/sdk/instrumentation#observe-wrapper) de Langfuse SDK v4 es la forma principal de instrumentar código Python. Lo que hace automáticamente:

| Aspecto | Comportamiento |
|---------|---------------|
| **Crea** | Un trace (si es la función raíz) o un span (si se llama dentro de otro `@observe`) |
| **Captura input** | Todos los parámetros de la función, serializados automáticamente |
| **Captura output** | El valor de retorno de la función |
| **Mide duración** | Desde la invocación hasta el return (incluye await en async) |
| **Registra errores** | Si la función lanza una excepción, se registra como error en el span |
| **Anida automáticamente** | Si `@observe` A llama a `@observe` B, B se anida como hijo de A |

```mermaid
graph TD
    T["Trace (se crea para la función raíz)"]
    T --> S["Span (@observe crea una observation)"]
    S --> SS["Span hijo (función @observe llamada dentro)"]
```

### Importación correcta en v4

```python
# ✅ Correcto — Langfuse v4
from langfuse import observe

# ❌ Incorrecto — era v3, ya no existe
# from langfuse.decorators import observe
```

### Parámetro `name` — siempre obligatorio en nuestro proyecto

```python
@observe(name="descriptive_name")  # Buena práctica — filtrable en dashboard
@observe()                          # Nombre auto-generado, difícil de filtrar
```

Convención de nombres: `<categoría>_<acción>` → `tool_search_catalog`, `scan_input`, `agent_process_query`

> **Referencia:** [Langfuse — Observe Wrapper](https://langfuse.com/docs/observability/sdk/instrumentation#observe-wrapper)

In [4]:
from langfuse import observe
import time


@observe(name="mi_primera_funcion")
def mi_funcion(x: int) -> int:
    """Una función simple instrumentada con Langfuse."""
    time.sleep(0.1)  # Simular trabajo
    return x * 2

# Ejecutar
resultado = mi_funcion(21)
print(f"Resultado: {resultado}")

# IMPORTANTE: en notebooks, hacemos flush manual para enviar las trazas
langfuse_client.flush()
print("\n-> Ve a Langfuse > Traces y busca 'mi_primera_funcion'")
print("  Verás: timestamp, duración, input (21), output (42)")

Resultado: 42

-> Ve a Langfuse > Traces y busca 'mi_primera_funcion'
  Verás: timestamp, duración, input (21), output (42)


**¿Qué acaba de pasar?**

1. `@observe(name="mi_primera_funcion")` interceptó la llamada a `mi_funcion(21)`
2. Langfuse registró: input=`21`, output=`42`, duración ≈ 100ms
3. Creó un **trace** (porque es la función raíz) con un **span** dentro
4. La información está en el buffer local — `lf_client.flush()` la envió al servidor

Ve a **Langfuse -> Traces** y busca el trace con nombre `mi_primera_funcion`. Verás el timestamp, duración, input y output.

> **Nótese:** Con una sola línea de código (`@observe`) obtuvimos trazabilidad completa. Este es el valor de un SDK de observabilidad bien diseñado.

### Trazas anidadas — jerarquía automática

Cuando una función `@observe` llama a otra `@observe`, Langfuse crea automáticamente una **jerarquía padre->hijo**. Esto es lo que nos permite ver el árbol completo de operaciones de un agente.

In [5]:
@observe(name="paso_validar")
def validar(texto: str) -> bool:
    """Simula una validación de input."""
    time.sleep(0.05)
    return len(texto) > 0


@observe(name="paso_procesar")
def procesar(texto: str) -> str:
    """Simula procesamiento."""
    time.sleep(0.1)
    return texto.upper()


@observe(name="pipeline_completo")
def pipeline(texto: str) -> str:
    """Pipeline que orquesta validación + procesamiento.

    En Langfuse verás la jerarquía:
    pipeline_completo (trace)
    -> paso_validar (span hijo)
    -> paso_procesar (span hijo)
    """
    if validar(texto):
        return procesar(texto)
    return "Input inválido"


resultado = pipeline("hola mundo")
print(f"Resultado: {resultado}")

langfuse_client.flush()
print("\n-> Ve a Langfuse > Traces > 'pipeline_completo'")
print("  Verás el árbol: pipeline_completo -> paso_validar + paso_procesar")

Resultado: HOLA MUNDO

-> Ve a Langfuse > Traces > 'pipeline_completo'
  Verás el árbol: pipeline_completo -> paso_validar + paso_procesar


---

## Parte 3: Instrumentar el agente TechShop

### 3.1 Importar el agente empaquetado

A partir de ahora usamos el agente de `src/techshop_agent/` — el mismo que construimos en el NB 1, pero empaquetado con búsqueda robusta y normalización Unicode.

In [6]:
from techshop_agent import create_agent

agent = create_agent()

print("[OK] Agente TechShop importado desde el paquete")

[OK] Agente TechShop importado desde el paquete


### 3.2 Instrumentar con `@observe` + `propagate_attributes`

Creamos una función wrapper que envuelve la llamada al agente con trazabilidad completa. Además de la captura automática de `@observe`, usamos **`propagate_attributes`** para propagar metadata a todo el árbol de trazas:

**¿Por qué `propagate_attributes` y no pasar metadata directamente?**

El problema: Strands Agents y Bedrock emiten spans OpenTelemetry internos (LLM calls, tool calls). Tú no controlas esas funciones — no puedes ponerles `@observe`. Pero sí necesitas que esos spans lleven tu `user_id` y `session_id`.

La solución: `propagate_attributes` establece un contexto que se hereda automáticamente por **todos los spans hijos** — tanto los que tú creas con `@observe` como los que emite Strands/Bedrock vía OTEL.

| Metadata | Para qué sirve en producción |
|----------|------------------------------|
| `user_id` | Agrupar queries por usuario, detectar patrones individuales |
| `session_id` | Reconstruir conversaciones completas en orden |
| `metadata.source` | Saber de dónde vino la query (web, API, notebook) |
| `metadata.query_length` | Correlacionar longitud de query con calidad de respuesta |

> **En Langfuse SDK v4**, `propagate_attributes` reemplaza al antiguo `langfuse_context.update_current_trace()` de la v3. Es el mecanismo correcto para propagar metadata en el SDK actual.

> **Referencia:** [Langfuse — Instrumentation](https://langfuse.com/docs/observability/sdk/instrumentation)

In [7]:
# Langfuse v4: propagate_attributes() propaga atributos a la traza Y todas sus observaciones hijas
from langfuse import observe, propagate_attributes

# langfuse ya fue inicializado con get_client() en la Parte 1 (singleton)


@observe(name="techshop_query", as_type="agent")
def process_query(user_query: str, user_id: str = "student", session_id: str = "nb02") -> str:
    """Procesa una consulta al agente con trazabilidad Langfuse.

    Langfuse captura automáticamente:
    - Timestamp de inicio y fin
    - Input (user_query) y Output (respuesta)
    - Duración total
    - Observaciones hijas (tool calls, LLM calls de Strands via OTEL)
    """
    # propagate_attributes() establece user_id, session_id y metadata en la traza activa
    # y los propaga automáticamente a TODAS las observaciones anidadas (tools, LLM calls).
    # IMPORTANTE: los valores de metadata deben ser strings (máx. 200 caracteres).
    # Llámalo al principio para que todas las observaciones hijas hereden los atributos.
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        metadata={"source": "notebook_02", "query_length": str(len(user_query))},
    ):
        response = agent(user_query)
        response_str = str(response)

        # update_current_span() añade metadata adicional a la observación actual
        # DESPUÉS de tener el resultado (por eso está aquí, no en propagate_attributes)
        langfuse_client.update_current_span(
            metadata={
                "response_length": str(len(response_str)),
                "response_word_count": str(len(response_str.split())),
            }
        )

        return response_str


print("[OK] Función process_query instrumentada con Langfuse v4")

[OK] Función process_query instrumentada con Langfuse v4


**Análisis del código de instrumentación (Langfuse v4):**

1. **`@observe(name="techshop_query")`** — Crea un trace (porque es la función raíz). El `name` nos permite filtrar en el dashboard.
2. **`propagate_attributes(user_id=..., session_id=..., metadata=...)`** — Propaga los atributos a **todas** las observations hijas automáticamente, incluidas las que emite Strands Agents vía OTEL.
3. **`user_id` y `session_id`** — Son ciudadanos de primera clase en Langfuse: tienen filtros dedicados en el dashboard y métricas por usuario/sesión.
4. **`metadata`** — Los valores **deben ser strings** (máx. 200 chars). Usa `str(valor)` para convertir números.
5. **`lf_client.update_current_span(metadata={...})`** — Enriquece la observation activa con metadata adicional que solo existe después de ejecutar la lógica (ej. longitud de respuesta).

### Sesiones — Reconstruir conversaciones completas

Un usuario no hace una sola pregunta — tiene una conversación. Las sesiones agrupan múltiples trazas de la misma conversación:

```python
session_id = "session-nb02-demo"  # Mismo ID para toda la conversación

process_query("¿Qué portátiles tenéis?",   session_id=session_id)  # Trace 1
process_query("¿Y el más barato?",          session_id=session_id)  # Trace 2
process_query("¿Cuánto tarda el envío?",    session_id=session_id)  # Trace 3
```

En el dashboard de Langfuse, puedes filtrar por `session_id` para ver toda la conversación en orden, con coste total y latencia acumulada. Esto es fundamental para:
- **Depuración:** Ver toda la conversación de un usuario que reportó un problema
- **Análisis de UX:** ¿Cuántas preguntas necesita un usuario para obtener lo que busca?
- **Costes por conversación:** No solo por request, sino por sesión completa

### Instrumentar herramientas con `@observe`

En `src/techshop_agent/solution/observability.py`, cada herramienta tiene su propio span:

```python
@observe(name="tool_search_catalog")
def observed_search_catalog(query: str) -> str:
    result = _search_catalog_impl(query)
    lf_client.update_current_span(metadata={"results_count": str(len(products))})
    return result
```

Este patrón (separar instrumentación de lógica pura) permite testear la lógica con pytest sin Langfuse, y tener trazas completas en producción.

> **Cambio respecto a SDK v3:** En v3 se usaba `langfuse_context.update_current_trace(...)`. En v4, ese módulo no existe. Usa `propagate_attributes()` para atributos de traza y `lf_client.update_current_span()` / `lf_client.update_current_generation()` para actualizar observations.

### 3.3 Generar trazas con consultas diversas

Ejecutamos **8 consultas variadas** para generar un conjunto representativo de trazas. La variedad es intencionada:

| Queries 1-2 | Queries 3-4 | Queries 5-6 | Queries 7-8 |
|-------------|-------------|-------------|-------------|
| Productos directos | FAQs | Fuera de ámbito + ambigua | Combinadas |

Esto nos permitirá observar en el dashboard diferencias de **latencia**, **uso de herramientas** y **patrones de respuesta**.

In [8]:
queries = [
    "¿Qué portátiles tenéis disponibles?",
    "¿Cuál es la política de devoluciones?",
    "Quiero un móvil con buena cámara",
    "¿Hacéis envíos internacionales?",
    "¿Cuál es la capital de Francia?",
    "Necesito unos auriculares con cancelación de ruido",
    "¿Tenéis tablets?",
    "¿Puedo pagar en cuotas?",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'=' * 60}")
    print(f"Query {i}/{len(queries)}: {query}")
    print('=' * 60)
    try:
        response = process_query(
            user_query=query,
            user_id="student01",
            session_id="session-nb02-demo",
        )
        # Mostrar solo primeros 200 chars para no saturar el output
        preview = response[:200] + "..." if len(response) > 200 else response
        print(preview)
    except Exception as e:
        print(f"[ERROR] {e}")

# Flush para enviar todas las trazas
langfuse_client.flush()
print(f"\n\n[OK] {len(queries)} trazas enviadas a Langfuse")


Query 1/8: ¿Qué portátiles tenéis disponibles?

Tool #1: search_catalog
¡Hola! En TechShop tenemos dos excelentes portátiles disponibles:

1. **ProBook X1** - **$1.199,99**
   - Portátil profesional de 14 pulgadas
   - 16GB RAM y 512GB SSD
   - Stock disponible: 12 unidades
   - Perfecto para tareas profesionales exigentes

2. **NanoBook Air 13** - **$999,50**
   - Portátil ultraligero para trabajo diario
   - Batería de larga duración
   - Stock disponible: 8 unidades
   - Ideal si buscas portabilidad y rendimiento equilibrado

¿Te gustaría conocer más detalles de alguno de estos modelos o tienes alguna necesidad específica que pueda ayudarte a elegir?¡Hola! En TechShop tenemos dos excelentes portátiles disponibles:

1. **ProBook X1** - **$1.199,99**
   - Portátil profesional de 14 pulgadas
   - 16GB RAM y 512GB SSD
   - Stock disponible: 12 unidad...

Query 2/8: ¿Cuál es la política de devoluciones?

Tool #2: get_faq_answer
Claro, aquí está nuestra **política de devoluciones**:

✅ *

---

## Parte 4: Analizar trazas en el dashboard

Ve a **[cloud.langfuse.com](https://cloud.langfuse.com)** -> tu proyecto -> **Traces**.

### Guía de exploración paso a paso

**1. Vista de Traces (lista)**
- Cada fila = una consulta del usuario
- Columnas: nombre, duración, user_id, timestamp
- Filtra por `user_id = student01` para ver solo tus trazas

**2. Vista detallada (click en una traza)**
- Verás el **árbol de spans**: qué operaciones se ejecutaron
- Input del usuario (la query original)
- Output del agente (la respuesta generada)
- Metadata que añadimos (`source`, `query_length`)

**3. Preguntas de análisis**

| Pregunta | Dónde buscar | Qué implica |
|----------|-------------|-------------|
| ¿Hay queries donde el agente NO usó herramientas? | Árbol de spans (traces sin tool calls) | Posible alucinación |
| ¿Alguna query tardó mucho más? | Columna Duration | Cuello de botella |
| ¿Respondió preguntas fuera de ámbito? | Output de la query "capital de Francia" | Falta de guardrails |
| ¿El session_id agrupa bien? | Filtro Session | Reconstrucción de conversaciones |

> **Referencia:** [Langfuse — Traces UI](https://langfuse.com/docs/tracing)

---

## Parte 5: Consultar trazas por API

El dashboard es útil para exploración manual, pero en LLMOps necesitamos **análisis programático**. Langfuse no es solo un dashboard visual — su API permite consultar trazas programáticamente para alertas, análisis y generación de datasets.

### Métodos clave de `lf_client.api` (SDK v4)

| Método | Qué devuelve | Uso típico |
|--------|-------------|-----------|
| `lf_client.api.trace.list(limit=N, user_id=...)` | Lista paginada de trazas | Análisis de latencia, errores |
| `lf_client.api.trace.get(trace_id)` | Una traza completa | Debug de una request específica |
| `lf_client.api.observations.get_many(trace_id=id)` | Observations de un trace | Ver qué tools se llamaron |
| `lf_client.api.sessions.list(limit=N)` | Sesiones agrupadas | Reconstruir conversaciones |

### Casos de uso de la API
- **Alertas automáticas:** Detectar trazas con coste > $0.05 y notificar por Slack
- **Análisis de calidad:** Extraer trazas con score bajo para revisión humana
- **Datasets de evaluación:** Seleccionar trazas reales como test cases para CI
- **Auditoría:** Verificar que no haya respuestas fuera de alcance en las últimas 24h

> **Cambio respecto a v3:** Los métodos `langfuse.fetch_traces()`, `langfuse.fetch_observations()` y `lf_client.score()` del SDK v3 ya **no existen** en v4. Ahora se usa `lf_client.api.trace.list()`, `lf_client.api.observations.get_many()` y `langfuse.create_score()`.

> **Referencia:** [Langfuse — Query Data via SDKs](https://langfuse.com/docs/api-and-data-platform/features/query-via-sdk)

In [9]:
# Consultar las últimas trazas via lf_client.api (SDK v4)
# Nota: los datos nuevos pueden tardar 15-30 segundos en aparecer en la API
try:
    result = langfuse_client.api.trace.list(limit=10, user_id="student01")
    traces_data = result.data

    print(f"Últimas {len(traces_data)} trazas del usuario 'student01':\n")
    for t in traces_data:
        latency_str = f"{t.latency * 1000:.0f}ms" if getattr(t, "latency", None) else "N/A"
        user_str = (getattr(t, "user_id", None) or "N/A")
        name_str = (getattr(t, "name", None) or "unnamed")[:25]
        print(f"  [{name_str:25s}] latency={latency_str:>8s} | user={user_str}")

except Exception as e:
    print(f"[ERROR] No se pudieron obtener trazas: {e}")
    print("   Asegúrate de haber ejecutado las queries de la Parte 3 primero.")
    # Por si acaso la API de query aún no está disponible, muestra instrucciones
    print("\n   Alternativa: Ve directamente a cloud.langfuse.com > Traces")

Últimas 10 trazas del usuario 'student01':

  [unnamed                  ] latency=  2066ms | user=student01
  [techshop_query           ] latency=  2851ms | user=student01
  [unnamed                  ] latency=  4875ms | user=student01
  [techshop_query           ] latency=  3356ms | user=student01
  [techshop_query           ] latency=  5134ms | user=student01
  [techshop_query           ] latency=  4035ms | user=student01
  [techshop_query           ] latency=  2873ms | user=student01
  [techshop_query           ] latency=  3234ms | user=student01
  [techshop_query           ] latency=  2379ms | user=student01
  [techshop_query           ] latency=  3308ms | user=student01


In [10]:
# Análisis más profundo: ver las observaciones de un trace específico
try:
    result = langfuse_client.api.trace.list(limit=5, user_id="student01")
    if result.data:
        first_trace = result.data[0]
        print(f"Detalle del trace: {getattr(first_trace, 'name', 'N/A')} (id: {first_trace.id[:12]}...)")
        print(f"   User: {getattr(first_trace, 'user_id', 'N/A')}")
        print(f"   Session: {getattr(first_trace, 'session_id', 'N/A')}")
        latency = getattr(first_trace, "latency", None)
        print(f"   Latency: {latency * 1000:.0f}ms" if latency else "   Latency: N/A")

        # Obtener las observaciones (spans, generations) de este trace
        obs_result = langfuse_client.api.observations.get_many(
            trace_id=first_trace.id,
            limit=20,
        )
        obs_data = obs_result.data
        print(f"\n   Observaciones ({len(obs_data)} en total):")
        for obs in obs_data:
            obs_type = getattr(obs, "type", "span") or "span"
            obs_name = (getattr(obs, "name", None) or "unnamed")[:30]
            obs_latency = getattr(obs, "latency", None)
            duration = f"{obs_latency * 1000:.0f}ms" if obs_latency else "N/A"
            print(f"      [type={obs_type:12s}] {obs_name:30s} | duration={duration}")
    else:
        print("No se encontraron trazas. Ejecuta la celda de queries primero.")
except Exception as e:
    print(f"[ERROR] {e}")
    print("   Ejecuta las queries de la Parte 3 y espera ~30 segundos antes de consultar.")

Detalle del trace:  (id: 5024eadc8be0...)
   User: student01
   Session: session-nb02-demo
   Latency: 2066ms

   Observaciones (4 en total):
      [type=TOOL        ] search_catalog                 | duration=4ms
      [type=GENERATION  ] chat                           | duration=1050ms
      [type=TOOL        ] search_catalog                 | duration=N/A
      [type=GENERATION  ] chat                           | duration=917ms


---

## Parte 6: Las 4 categorías de métricas LLMOps

Con trazas en Langfuse, obtienes métricas que sin observabilidad son completamente invisibles.

### Las 4 categorías

| Categoría | Qué medimos | Ejemplo real TechShop | Herramientas |
|-----------|-------------|----------------------|-------------|
| **Operacionales** | Latencia (P50, P95, P99), tasa de error, throughput | P50: 1.2s, P95: 3.1s, errores: 0.3% | Langfuse, CloudWatch |
| **Coste** | $/request, $/sesión, $/usuario, $/día, tokens totales | Promedio: $0.003/request, $15/día | Langfuse (tokens) |
| **Calidad** | Scores de accuracy, relevance, hallucination rate | 92% accuracy, 3% alucinación | Evaluaciones (NB 4) |
| **Uso** | Queries por categoría, herramientas más usadas, horas pico | 40% catálogo, 30% FAQ, pico a las 10:00 | Langfuse analytics |


> **Insight clave:** Las categorías 1 y 2 (operacionales, coste) se pueden medir con herramientas de infraestructura clásicas. Las categorías 3 y 4 (calidad, uso) **necesitan herramientas LLMOps** — por eso Langfuse es tan importante.

```mermaid
flowchart TD
    subgraph Infra["Métricas Operacionales"]
        CW["CloudWatch / Datadog<br/>Latencia P50/P99<br/>Error rate<br/>Throughput"]
    end
    subgraph Calidad["Métricas de Calidad"]
        EV["Evaluaciones LLMOps<br/>Alucinaciones<br/>Relevancia<br/>Fidelidad"]
    end
    subgraph Puente["Langfuse — Puente entre infra y calidad"]
        LF["Trazas · Tokens/coste<br/>Prompts · Sesiones"]
    end

    CW --> LF
    EV --> LF
```

---

## Ejercicios opcionales

### Ejercicio 1: Añade metadata personalizado

Modifica `process_query` para capturar la **longitud de la respuesta** y el **número de palabras** como metadata adicional. Luego consulta las trazas por API y analiza la correlación entre longitud de query y longitud de respuesta.

> **Pista:** Usa `lf_client.update_current_span()` para añadir metadata al span actual.

In [ ]:
# === EJERCICIO 1: Metadata enriquecido (Langfuse v4) ===
# Descomenta y modifica para añadir más metadata post-respuesta

from langfuse import observe, propagate_attributes

@observe(name="techshop_query_enriched")
def process_query_enriched(user_query: str, user_id: str = "student", session_id: str = "nb02") -> str:
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        metadata={"source": "notebook_02", "query_length": str(len(user_query))},
    ):
        response = agent(user_query)
        response_str = str(response)

        # TODO: Añade metadata sobre la respuesta usando lf_client.update_current_span()
        # Los valores DEBEN ser strings (requisito de propagate_attributes / update_current_span)
        langfuse_client.update_current_span(
            metadata={
                "response_length": str(len(response_str)),
                "response_word_count": str(len(response_str.split())),
            }
        )

        return response_str

result = process_query_enriched("¿Qué portátiles tenéis?")
print(result[:200])
langfuse_client.flush()

### Ejercicio 2: Tags y metadata avanzado para clasificar trazas

La metadata que hemos visto hasta ahora (`query_length`, `response_length`) es un buen punto de partida, pero en produccion necesitas metadata que te permita **segmentar, filtrar y analizar** trazas de forma operativa.

Langfuse soporta tres mecanismos complementarios de enriquecimiento que conviene conocer:

| Mecanismo | Que captura | Donde se propaga | Caso de uso tipico |
|-----------|-------------|------------------|--------------------|
| **`metadata`** (key-value) | Datos estructurados por observacion | Via `propagate_attributes` a todo el arbol | Feature flags, version del modelo, region, longitud de query |
| **`tags`** (lista de strings) | Etiquetas categoricas filtrables | Via `propagate_attributes` a todo el arbol | Tipo de query, canal de entrada, experimento A/B |
| **`scores`** (numeric/boolean/categorical) | Metricas de calidad por traza u observacion | No se propagan -- se adjuntan a traza u observacion | Relevancia, satisfaccion del usuario, exito de herramienta |

**Ideas de metadata avanzado para un agente como TechShop:**

| Metadata / Tag | Tipo | Para que sirve en produccion |
|----------------|------|------------------------------|
| `intent_detected` | metadata | Clasificar queries por intencion (catalogo, FAQ, comparacion, stock) |
| `tools_used` | metadata | Saber que herramientas utilizo el agente en cada query |
| `tools_count` | metadata | Detectar queries que disparan demasiadas llamadas a herramientas |
| `empty_result` | metadata | Alertar cuando las herramientas no devuelven resultados utiles |
| `budget_range` | tag | Segmentar por rango de presupuesto del cliente |
| `channel` | tag | Segmentar trazas por canal: notebook, streamlit, API |
| `experiment_v2` | tag | Marcar trazas de un experimento A/B con nuevo prompt |
| `response_has_price` | metadata | Verificar que las recomendaciones incluyen precio (requisito del prompt) |

> **Referencia:** [Langfuse -- Metadata](https://langfuse.com/docs/observability/features/metadata) | [Langfuse -- Tags](https://langfuse.com/docs/observability/features/tags)

**Tu tarea:** Modifica `process_query_with_tags` para que:
1. Clasifique la query del usuario con tags basados en palabras clave
2. Añada metadata sobre la respuesta (si contiene un precio, longitud)
3. Propague todo con `propagate_attributes`

In [11]:
# === EJERCICIO 2: Tags y metadata avanzado ===

import re

from langfuse import observe, propagate_attributes


def classify_query_tags(query: str) -> list[str]:
    """Clasifica una query en tags basados en palabras clave."""
    tags = []
    q = query.lower()

    # Clasificar por intención detectada
    if any(w in q for w in ["compar", "versus", "vs", "diferencia entre"]):
        tags.append("intent:comparacion")
    elif any(w in q for w in ["stock", "disponib", "quedan", "hay"]):
        tags.append("intent:stock")
    elif any(w in q for w in ["recomien", "sugier", "presupuesto", "menos de"]):
        tags.append("intent:recomendacion")
    elif any(w in q for w in ["devoluci", "garant", "envio", "pago", "horario"]):
        tags.append("intent:faq")
    else:
        tags.append("intent:catalogo")

    # Tag de canal (en este caso, notebook)
    tags.append("channel:notebook")

    return tags


@observe(name="techshop_query_tagged")
def process_query_with_tags(
    user_query: str, user_id: str = "student", session_id: str = "nb02-ej2"
) -> str:
    """Query con tags e metadata avanzado."""
    # 1. Clasificar la query
    tags = classify_query_tags(user_query)

    # 2. Propagar atributos: metadata + tags se heredan en TODO el árbol
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=tags,
        metadata={
            "source": "notebook_02",
            "query_length": str(len(user_query)),
            "query_word_count": str(len(user_query.split())),
        },
    ):
        response = agent(user_query)
        response_str = str(response)

        # 3. Metadata post-respuesta: analizar el contenido generado
        has_price = bool(re.search(r"\d+[.,]\d{2}\s*€|€\s*\d+", response_str))

        langfuse_client.update_current_span(
            metadata={
                "response_length": str(len(response_str)),
                "response_has_price": str(has_price),
                "tags_applied": str(tags),
            }
        )

        return response_str


# --- Probar con queries de distintas intenciones ---
test_queries = [
    "¿Qué auriculares tenéis por menos de 150€?",
    "¿Cuál es la política de devoluciones?",
    "¿Hay stock del VoltPhone S?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    print(f"  Tags: {classify_query_tags(q)}")
    resp = process_query_with_tags(q)
    print(f"  Respuesta: {resp[:120]}...")

langfuse_client.flush()
print("\n-> Ve a Langfuse > Traces y filtra por tags 'intent:recomendacion', 'intent:faq', etc.")


Query: ¿Qué auriculares tenéis por menos de 150€?
  Tags: ['intent:recomendacion', 'channel:notebook']

Tool #9: search_catalog
Lo siento, en este momento **no tenemos auriculares disponibles en nuestro catálogo**, incluso en rango de precios por debajo de 150€. 😕

📢 **Lo que puedo hacer:**
- Revisar más tarde si tenemos nuevas llegadas
- Sugerirte que nos contactes directamente para conocer cuándo tendremos auriculares en stock
- Ayudarte a buscar otros productos electrónicos

¿Hay algo más en lo que pueda ayudarte? Podría buscarte otros accesorios o dispositivos que tengamos disponibles 😊  Respuesta: Lo siento, en este momento **no tenemos auriculares disponibles en nuestro catálogo**, incluso en rango de precios por d...

Query: ¿Cuál es la política de devoluciones?
  Tags: ['intent:faq', 'channel:notebook']

Tool #10: get_faq_answer
Claro, aquí está nuestra **política de devoluciones**:

✅ **Reembolsos:**
- Aceptamos solicitudes de reembolso dentro de los **30 días** posteriores a

### Ejercicio 3: Scores programaticos -- medir calidad en tiempo real

Los **scores** de Langfuse son el mecanismo para adjuntar metricas de calidad directamente a trazas u observaciones. A diferencia de metadata (informativo), un score es una **medida cuantificable** que aparece en dashboards, se puede agregar, y permite monitorizar calidad en el tiempo.

Langfuse soporta tres tipos de score:

| Tipo | Valores | Ejemplo TechShop |
|------|---------|------------------|
| **Numeric** | Float (ej: 0.0 -- 1.0) | Relevancia de la respuesta, latencia normalizada |
| **Boolean** | `True` / `False` | La respuesta incluyo precio? El agente uso herramientas? |
| **Categorical** | String (ej: "alta", "media", "baja") | Nivel de confianza, tipo de respuesta |

**Casos de uso reales para scores en agentes:**

| Score | Tipo | Logica | Valor en produccion |
|-------|------|--------|---------------------|
| `tool_success` | Boolean | La herramienta devolvio resultados? | Detectar "tool misses" -- queries que no matchean nada |
| `response_completeness` | Numeric | Ratio palabras respuesta / palabras query | Detectar respuestas demasiado cortas o verbose |
| `price_mentioned` | Boolean | Regex detecta precio en la respuesta | Verificar cumplimiento del prompt ("menciona su precio") |
| `multi_tool_used` | Boolean | Se usaron 2+ herramientas? | Medir complejidad de las queries |
| `user_satisfaction` | Categorical | Feedback del usuario (positivo/negativo/neutro) | Calidad percibida por el usuario |

```python
# Patron basico para adjuntar un score a la traza actual (Langfuse v4)
langfuse_client.score_current_trace(
    name="tool_success",
    value=True,
    data_type="BOOLEAN",
    comment="La herramienta devolvio al menos 1 resultado",
)

# Score numerico en una observacion especifica
langfuse_client.score_current_span(
    name="response_completeness",
    value=0.85,
    data_type="NUMERIC",
    comment="Ratio respuesta/query",
)
```

> **Referencia:** [Langfuse -- Scores via API/SDK](https://langfuse.com/docs/evaluation/evaluation-methods/scores-via-sdk)

**Tu tarea:** Implementa `process_query_with_scores` que:
1. Ejecute la query contra el agente
2. Adjunte un score booleano `price_mentioned` (la respuesta contiene un precio?)
3. Adjunte un score numerico `response_completeness` (ratio de longitud respuesta/query)

In [12]:
# === EJERCICIO 3: Scores programáticos ===

import re

from langfuse import observe, propagate_attributes


@observe(name="techshop_query_scored")
def process_query_with_scores(
    user_query: str, user_id: str = "student", session_id: str = "nb02-ej3"
) -> str:
    """Query con scores de calidad adjuntos a la traza."""
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        metadata={"source": "notebook_02"},
    ):
        response = agent(user_query)
        response_str = str(response)

        # --- Score 1: ¿La respuesta menciona un precio? (Boolean) ---
        has_price = bool(re.search(r"\d+[.,]\d{2}\s*€|€\s*\d+", response_str))
        langfuse_client.score_current_trace(
            name="price_mentioned",
            value=has_price,
            data_type="BOOLEAN",
            comment=f"Precio {'detectado' if has_price else 'ausente'} en la respuesta",
        )

        # --- Score 2: Completeness — ratio longitud respuesta / query (Numeric) ---
        completeness = min(len(response_str) / max(len(user_query), 1), 10.0)
        langfuse_client.score_current_trace(
            name="response_completeness",
            value=round(completeness, 2),
            data_type="NUMERIC",
            comment=f"Ratio respuesta/query: {completeness:.2f}",
        )

        return response_str


# --- Probar con queries donde esperamos diferente comportamiento ---
scored_queries = [
    ("¿Qué portátiles tenéis por menos de 1000€?", True),   # Esperamos precio
    ("¿Cuál es el horario de atención?", False),              # No esperamos precio
    ("Recomiéndame un smartphone gama alta", True),           # Esperamos precio
]

for q, expect_price in scored_queries:
    print(f"\nQuery: {q}")
    resp = process_query_with_scores(q)
    has_price = bool(re.search(r"\d+[.,]\d{2}\s*€|€\s*\d+", resp))
    status = "[OK]" if has_price == expect_price else "[MISS]"
    print(f"  {status} Precio esperado: {expect_price}, detectado: {has_price}")
    print(f"  Respuesta: {resp[:120]}...")

langfuse_client.flush()
print("\n-> Ve a Langfuse > Traces > pestaña 'Scores'")
print("  Verás 'price_mentioned' (boolean) y 'response_completeness' (numeric) por traza")


Query: ¿Qué portátiles tenéis por menos de 1000€?

Tool #12: search_catalog
Según nuestro catálogo, tenemos un portátil que se ajusta a tu presupuesto:

💻 **NanoBook Air 13** - **$999,50**
   - Portátil ultraligero para trabajo diario
   - Batería de larga duración
   - Stock disponible: 8 unidades
   - ⚠️ *Justo por debajo de los 1.000€*

El otro portátil que tenemos (**ProBook X1** a $1.199,99) supera tu presupuesto.

El **NanoBook Air 13** es una excelente opción si buscas portabilidad y buen rendimiento a un precio competitivo. ¿Te gustaría saber más detalles sobre este modelo? 😊  [MISS] Precio esperado: True, detectado: False
  Respuesta: Según nuestro catálogo, tenemos un portátil que se ajusta a tu presupuesto:

💻 **NanoBook Air 13** - **$999,50**
   - Po...

Query: ¿Cuál es el horario de atención?
Esa es una buena pregunta, pero lamentablemente **no tengo información sobre el horario de atención** de TechShop en mi base de datos. 😕

📞 **Te recomiendo que:**
- Visites nuestra p

---

### Solucion de referencia (Ejercicios 2 y 3 combinados)

<details>
<summary>Haz clic para ver la solucion completa</summary>

La solucion combina los tres mecanismos de enriquecimiento de Langfuse en una sola funcion:

1. **Tags** para clasificacion filtrable (intencion, canal, experimento)
2. **Metadata propagada** para contexto heredado por todo el arbol de trazas
3. **Metadata post-respuesta** para metricas calculadas sobre el output
4. **Scores** para metricas de calidad cuantificables y agregables

**Patron clave:** `propagate_attributes` se llama **al inicio** del bloque para que tags y metadata se hereden en todas las observaciones hijas (tool calls, LLM generations de Strands). Los scores y la metadata post-respuesta se adjuntan **despues** de tener el resultado.

```mermaid
flowchart TD
    Q["Query del usuario"] --> CL["Clasificar intent -> tags"]
    CL --> PA["propagate_attributes<br/>(tags + metadata iniciales)"]
    PA --> AG["agent(query)"]
    AG --> PM["update_current_span<br/>(metadata post-respuesta)"]
    PM --> SC["score_current_trace<br/>(scores de calidad)"]
    SC --> R["Retornar respuesta"]
```

El codigo de la solucion ejecutable esta en la celda siguiente.

</details>

In [13]:
# === SOLUCIÓN: Observabilidad avanzada — Tags + Metadata + Scores ===

import re

from langfuse import observe, propagate_attributes


# --- Clasificador de intención por palabras clave ---
INTENT_RULES: list[tuple[str, list[str]]] = [
    ("intent:comparacion", ["compar", "versus", "vs", "diferencia entre"]),
    ("intent:stock", ["stock", "disponib", "quedan", "hay"]),
    ("intent:recomendacion", ["recomien", "sugier", "presupuesto", "menos de"]),
    ("intent:faq", ["devoluci", "garant", "envio", "pago", "horario", "soporte"]),
]


def classify_intent(query: str) -> list[str]:
    """Clasifica la query en tags de intención."""
    q = query.lower()
    tags = ["channel:notebook"]
    matched = False
    for tag, keywords in INTENT_RULES:
        if any(kw in q for kw in keywords):
            tags.append(tag)
            matched = True
            break
    if not matched:
        tags.append("intent:catalogo")
    return tags


@observe(name="techshop_query_full_observability")
def process_query_full(
    user_query: str,
    user_id: str = "student",
    session_id: str = "nb02-solution",
) -> str:
    """Query con observabilidad completa: tags + metadata + scores."""

    # --- 1. Clasificar y propagar al inicio ---
    tags = classify_intent(user_query)

    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=tags,
        metadata={
            "source": "notebook_02",
            "query_length": str(len(user_query)),
            "query_word_count": str(len(user_query.split())),
        },
    ):
        # --- 2. Ejecutar el agente ---
        response = agent(user_query)
        response_str = str(response)

        # --- 3. Metadata post-respuesta ---
        has_price = bool(re.search(r"\d+[.,]\d{2}\s*€|€\s*\d+", response_str))
        langfuse_client.update_current_span(
            metadata={
                "response_length": str(len(response_str)),
                "response_word_count": str(len(response_str.split())),
                "response_has_price": str(has_price),
                "intent_tags": str(tags),
            }
        )

        # --- 4. Scores de calidad ---
        # Score booleano: ¿la respuesta incluye precio? (requisito del system prompt)
        langfuse_client.score_current_trace(
            name="price_mentioned",
            value=has_price,
            data_type="BOOLEAN",
            comment=f"Precio {'detectado' if has_price else 'ausente'}",
        )

        # Score numérico: completeness (ratio longitud respuesta / query, capped a 10)
        completeness = min(len(response_str) / max(len(user_query), 1), 10.0)
        langfuse_client.score_current_trace(
            name="response_completeness",
            value=round(completeness, 2),
            data_type="NUMERIC",
            comment=f"Ratio: {completeness:.2f}",
        )

        return response_str


# --- Demostración con distintos tipos de query ---
demo_queries = [
    "¿Qué portátiles tenéis por menos de 1000€?",
    "¿Cuál es la política de devoluciones?",
    "¿Hay stock del VoltPhone S?",
]

for q in demo_queries:
    tags = classify_intent(q)
    print(f"\nQuery: {q}")
    print(f"  Tags: {tags}")
    resp = process_query_full(q)
    print(f"  Respuesta: {resp[:150]}...")

langfuse_client.flush()

print("\n" + "=" * 70)
print("Ve a Langfuse y explora:")
print("  1. Traces > filtra por tag 'intent:recomendacion' o 'intent:faq'")
print("  2. Traces > columna Scores: veras 'price_mentioned' y 'response_completeness'")
print("  3. Trace detalle > Metadata: veras query_length, response_has_price, etc.")
print("  4. Score Analytics: agrega 'price_mentioned' para ver % de respuestas con precio")


Query: ¿Qué portátiles tenéis por menos de 1000€?
  Tags: ['channel:notebook', 'intent:recomendacion']
Basándome en nuestra búsqueda anterior, tenemos un portátil que se ajusta a tu presupuesto:

💻 **NanoBook Air 13** - **$999,50**
   - Portátil ultraligero ideal para trabajo diario
   - Batería de larga duración
   - Stock disponible: 8 unidades
   - ✅ Dentro de tu presupuesto de menos de 1.000€

Este es el único portátil que tenemos por debajo de los 1.000€. Es una excelente opción si buscas portabilidad, ligereza y buen rendimiento para tareas cotidianas.

El otro modelo que disponemos (**ProBook X1** a $1.199,99) supera tu presupuesto.

¿Te gustaría conocer más detalles técnicos del **NanoBook Air 13** o tienes alguna otra pregunta? 😊  Respuesta: Basándome en nuestra búsqueda anterior, tenemos un portátil que se ajusta a tu presupuesto:

💻 **NanoBook Air 13** - **$999,50**
   - Portátil ultrali...

Query: ¿Cuál es la política de devoluciones?
  Tags: ['channel:notebook', 'intent:f

---

## Resumen y conceptos clave

### Antes vs. despues

| Sin observabilidad (NB 1) | Con observabilidad (NB 2) |
|---------------------------|---------------------------|
| "El agente respondio algo" | Veo exactamente que queries recibio |
| "Parece que funciona" | Conozco la latencia de cada llamada |
| "No se si uso las herramientas" | Veo el arbol completo de observations |
| "Cuanto cuesta?" | Tokens y coste por request |
| "Quien pregunto que?" | Trazas filtradas por user_id y session_id |
| "Que tipo de queries llegan?" | Tags por intencion + metadata de canal |
| "La respuesta fue buena?" | Scores booleanos y numericos por traza |

### Errores comunes y como evitarlos

| Problema | Causa | Solucion |
|----------|-------|----------|
| **Trazas vacias** | `@observe` no captura los spans de Strands/Bedrock | Verificar que `OTEL_EXPORTER_OTLP_ENDPOINT` apunta a Langfuse |
| **user_id/session_id falta** | No se uso `propagate_attributes()` | Envolver la llamada al agente con `propagate_attributes()` |
| **Trazas no aparecen** | Las credenciales de Langfuse son incorrectas | Ejecutar `lf_client.auth_check()` al inicio |
| **SDK v3 vs v4** | Codigo copiado de tutorial antiguo | Usar `get_client()`, `propagate_attributes()`, `lf_client.api.trace.list()` |
| **Tags no se propagan** | `propagate_attributes` llamado tarde | Llamar `propagate_attributes` lo antes posible en la traza |
| **Scores no aparecen** | Falta `flush()` o `trace_id` incorrecto | Usar `score_current_trace()` dentro del bloque `@observe` |

### Conceptos que debes llevarte

1. **Los logs no son suficientes** para agentes LLM -- necesitas trazas que muestren el flujo completo
2. **Trace -> Span -> Generation -> Event** es el modelo de datos de Langfuse
3. **`@observe`** captura automaticamente input, output, duracion y errores
4. **`propagate_attributes`** propaga user_id, session_id, metadata y tags a todo el arbol de trazas
5. **Las sesiones** agrupan trazas de una misma conversacion
6. **Tags** permiten clasificar trazas por intencion, canal o experimento -- filtrables en el dashboard
7. **Metadata** enriquece observaciones con contexto estructurado (key-value, strings max 200 chars)
8. **Scores** (numeric, boolean, categorical) cuantifican calidad y son agregables en Score Analytics
9. **4 categorias de metricas**: operacionales, coste, calidad, uso
10. **La API de Langfuse** permite consultar trazas programaticamente para alertas, analisis y datasets
11. **`lf_client.flush()`** en notebooks siempre necesario antes de ir al dashboard

### Implementacion de referencia

El archivo `src/techshop_agent/solution/observability.py` contiene la implementacion completa del agente instrumentado con 5 herramientas observadas. El patron: `@strands_tool` (interfaz) -> `@observe` (instrumentacion) -> `_impl()` (logica pura testeable). Cada herramienta muestra un patron de metadata diferente:

| Herramienta | Patron de tracing |
|-------------|-------------------|
| `search_catalog` | Conteo de resultados + query normalizada |
| `get_faq_answer` | Flag booleano de match + topic |
| `compare_products` | Multi-input + flag de error |
| `check_stock` | Outcome categorico + valor numerico |
| `get_product_recommendations` | Filtros numericos + estadisticas del resultado |

### Referencias del notebook

- [Langfuse -- Documentation](https://langfuse.com/docs)
- [Langfuse -- Instrumentation (observe, propagate\_attributes)](https://langfuse.com/docs/observability/sdk/instrumentation)
- [Langfuse -- Tracing Data Model](https://langfuse.com/docs/observability/data-model)
- [Langfuse -- Metadata](https://langfuse.com/docs/observability/features/metadata)
- [Langfuse -- Tags](https://langfuse.com/docs/observability/features/tags)
- [Langfuse -- Scores via API/SDK](https://langfuse.com/docs/evaluation/evaluation-methods/scores-via-sdk)
- [Langfuse -- Query Data via SDKs](https://langfuse.com/docs/api-and-data-platform/features/query-via-sdk)
- [Langfuse -- Python v3 -> v4 Migration Guide](https://langfuse.com/docs/observability/sdk/upgrade-path/python-v3-to-v4)
- [OpenTelemetry -- Semantic Conventions for GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
- [Strands Agents -- Observability](https://strandsagents.com/latest/user-guide/concepts/agents/)

---

## Siguiente: [Notebook 3 -- Prompt Management con Langfuse](./03_prompt_management.ipynb)